# SPY 0DTE — 15-min Midpoint Pullback (Profit × Time-stop sweep)

Backtests a 0DTE SPY strategy on real Polygon data, **2024-06-01 → 2026-06-01**, then sweeps profit-target × time-stop.

## Strategy
- Work off the **15-minute** SPY chart.
- Entry is only considered from **10:30 ET onward** (= **7:30 AM Pacific**; change `ENTRY_START_ET` if you meant a different zone).
- Look at the **15-min candle that just closed** and take its **midpoint = (high + low) / 2**.
- Track SPY; when price **hits/crosses that midpoint**, enter immediately by buying a **0DTE ATM option**:
  - previous candle **green** → buy a **call**; **red** → buy a **put**.
- The reference candle rolls forward every 15 min; we take the **first** midpoint cross of the day → **one trade per day**.
- Exit: **take-profit at +X%** of premium if hit, otherwise **close at market after the time stop** (45 or 60 min), and in any case by EOD (~15:55). No percentage stop loss — the time stop is the risk control.

## Costs
- **Commission** $0.65 / contract on **both** sides.
- **Slippage**: the option must trade **$0.01 past** the take-profit for the limit to fill (profit measured **after** slippage + fees).

## Sweep
Profit targets **{2, 3, 5, 7, 10}%** × time stops **{45, 60} min** → 10 combos. Reports a final-balance matrix, a heatmap, and a full breakdown of the best combo.

Starting balance **$1,000**, full balance deployed each day (whole contracts).

> ⚠️ Modeled fills (resting-order prices + 1¢ slippage). Real 0DTE spreads make a 2–10% target optimistic. All-in on one 0DTE option daily is extreme risk. Research only.

In [ ]:
import requests, time, math
import datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Configuration

In [ ]:
import datetime as dt   # so this cell works even if run before the imports cell

# ---- API ----
POLYGON_API_KEY = "YOUR_POLYGON_API_KEY"   # <-- paste your Polygon key here in Colab

# ---- Backtest window ----
START_DATE = dt.date(2024, 6, 1)
END_DATE   = dt.date(2026, 6, 1)

# ---- Strategy ----
START_BALANCE  = 1_000.0
CANDLE_MIN     = 15            # reference candle size (minutes)
ENTRY_START_ET = "10:30"      # earliest entry; 10:30 ET == 7:30 AM Pacific
EOD_ET         = "15:55"      # force close before the bell

# ---- Costs ----
COMMISSION = 0.65             # $ per contract, charged on BOTH entry and exit
SLIPPAGE   = 0.01            # extra $ past target needed to fill the profit order

# ---- Sweep grids ----
PROFIT_GRID    = [0.02, 0.03, 0.05, 0.07, 0.10]   # take-profit % of premium
TIME_STOP_GRID = [45, 60]                          # exit at market after N minutes if profit not hit

# ---- API politeness (0.0 for unlimited plans) ----
REQUEST_SLEEP = 0.0

assert POLYGON_API_KEY != "YOUR_POLYGON_API_KEY", "Set your Polygon API key first!"
print(f"{START_DATE} -> {END_DATE} | entry from {ENTRY_START_ET} ET")

## 2. Polygon helpers

In [ ]:
BASE = "https://api.polygon.io"
SESSION = requests.Session()

def _get(url, params=None, max_retries=6):
    params = dict(params or {}); params["apiKey"] = POLYGON_API_KEY
    backoff = 1.0
    for _ in range(max_retries):
        try:
            r = SESSION.get(url, params=params, timeout=30)
        except requests.RequestException:
            time.sleep(backoff); backoff *= 2; continue
        if r.status_code == 200:
            if REQUEST_SLEEP: time.sleep(REQUEST_SLEEP)
            return r.json()
        if r.status_code in (401, 403):
            raise RuntimeError(f"{r.status_code} from Polygon: {r.text[:200]} "
                               "-> key lacks OPTIONS data access for this request.")
        time.sleep(backoff); backoff *= 2     # 429 / 5xx
    raise RuntimeError(f"Failed after retries: {url}")

def get_aggs(ticker, mult, timespan, date_str):
    url = f"{BASE}/v2/aggs/ticker/{ticker}/range/{mult}/{timespan}/{date_str}/{date_str}"
    j = _get(url, {"adjusted": "true", "sort": "asc", "limit": 50000})
    res = j.get("results")
    if not res: return None
    df = pd.DataFrame(res).rename(columns={"o":"open","h":"high","l":"low","c":"close","v":"vol"})
    df["t"] = pd.to_datetime(df["t"], unit="ms", utc=True).dt.tz_convert("America/New_York")
    return df.set_index("t").sort_index()

def get_trading_days(start, end):
    j = _get(f"{BASE}/v2/aggs/ticker/SPY/range/1/day/{start}/{end}",
             {"adjusted":"true","sort":"asc","limit":50000})
    return [pd.to_datetime(r["t"], unit="ms", utc=True).tz_convert("America/New_York").date()
            for r in j.get("results", [])]

def occ_ticker(expiry, cp, strike):
    return f"O:SPY{expiry:%y%m%d}{cp}{int(round(strike*1000)):08d}"

## 3. Phase 1 — find each day's entry & pull the option path
Entry detection (midpoint cross) and the contract chosen are **independent** of the profit/stop levels, so we fetch once per day and replay the sweep offline.

~3 API calls per trading day.

In [ ]:
def build_setup(day):
    """For one day: find first 15-min-midpoint cross from ENTRY_START_ET, buy 0DTE ATM,
       return entry premium + the option's intraday high/low/close path. None if no trade."""
    ds = day.isoformat()

    c = get_aggs("SPY", CANDLE_MIN, "minute", ds)
    if c is None: return None
    c = c.between_time("09:30", "16:00")
    if c.empty: return None
    c = c.copy()
    c["close_time"] = c.index + pd.Timedelta(minutes=CANDLE_MIN)
    c["mid"]   = (c["high"] + c["low"]) / 2.0
    c["green"] = c["close"] >= c["open"]

    m1 = get_aggs("SPY", 1, "minute", ds)
    if m1 is None: return None
    m1 = m1.between_time(ENTRY_START_ET, EOD_ET)
    if m1.empty: return None

    entry_time = mid = green = None
    for t, bar in m1.iterrows():
        prior = c[c["close_time"] <= t]
        if prior.empty: continue
        ref = prior.iloc[-1]
        if bar["low"] <= ref["mid"] <= bar["high"]:     # price touched the midpoint
            entry_time, mid, green = t, float(ref["mid"]), bool(ref["green"])
            break
    if entry_time is None: return None

    cp     = "C" if green else "P"
    strike = float(round(mid))                          # ATM, nearest $1
    opt = get_aggs(occ_ticker(day, cp, strike), 1, "minute", ds)
    if opt is None: return None
    opt = opt[(opt.index >= entry_time)].between_time(ENTRY_START_ET, EOD_ET)
    if opt.empty: return None
    entry = float(opt.iloc[0]["open"])
    if not entry or entry <= 0: return None

    return dict(date=day, color=("green" if green else "red"), cp=cp, strike=strike,
                entry=entry, highs=opt["high"].to_numpy(),
                lows=opt["low"].to_numpy(), closes=opt["close"].to_numpy())


days = get_trading_days(START_DATE, END_DATE)
print(f"{len(days)} trading days {days[0]} -> {days[-1]}\\n")

setups, skipped = [], 0
for i, d in enumerate(days, 1):
    try:
        s = build_setup(d)
    except RuntimeError as e:
        if "OPTIONS data access" in str(e): raise
        s = None
    if s is None:
        skipped += 1
    else:
        setups.append(s)
    if i % 50 == 0:
        print(f"{d}  scanned={i}  setups={len(setups)}")

print(f"\\nPhase 1 done: {len(setups)} tradeable days, {skipped} skipped.")

## 4. Phase 2 — replay the profit × time-stop sweep (offline, no API)

In [ ]:
def simulate(setups, pt, tstop):
    """Full-balance compounding replay: take-profit at +pt (after slippage+fees),
       otherwise exit at market after `tstop` minutes (time stop), else EOD."""
    bal, rows = START_BALANCE, []
    for s in setups:
        entry = s["entry"]
        target_fill = entry * (1 + pt)
        target_trig = target_fill + SLIPPAGE         # need +1c to fill (slippage)
        H, C = s["highs"], s["closes"]

        limit = min(tstop, len(H) - 1)               # bar index for the time stop
        exit_price = reason = None
        for i in range(limit + 1):
            if H[i] >= target_trig:                  # profit hit on/before the time stop
                exit_price, reason = target_fill, "target"; break
        if exit_price is None:                        # time stop (or EOD if data ran out first)
            exit_price = float(C[limit])
            reason = "time" if limit == tstop else "eod"

        contracts = int(bal // (entry * 100 + COMMISSION))
        if contracts < 1:                             # can't afford a contract -> no trade
            continue
        pnl = contracts * 100 * (exit_price - entry) - contracts * COMMISSION * 2
        bal += pnl
        rows.append(dict(date=s["date"], color=s["color"], cp=s["cp"], strike=s["strike"],
                         entry=entry, exit=exit_price, reason=reason, contracts=contracts,
                         pnl=pnl, ret=pnl / (contracts * entry * 100), balance=bal))
    return pd.DataFrame(rows)

results = {(pt, ts): simulate(setups, pt, ts) for ts in TIME_STOP_GRID for pt in PROFIT_GRID}
final_of = lambda df: (df["balance"].iloc[-1] if len(df) else START_BALANCE)

# ---- final-balance matrix (rows = time stop, cols = profit target) ----
matrix = pd.DataFrame(
    {f"PT {int(pt*100)}%": {f"{ts}min": final_of(results[(pt, ts)])
                            for ts in TIME_STOP_GRID} for pt in PROFIT_GRID})
print("Final balance — start $1,000 | rows = time stop, cols = profit target\n")
print(matrix.applymap(lambda v: f"${v:,.0f}").to_string())

best = max(results, key=lambda k: final_of(results[k]))
print(f"\nBest combo: +{int(best[0]*100)}% profit / {best[1]}min time stop "
      f"-> ${final_of(results[best]):,.2f}")

## 5. Heatmap of final balances

In [ ]:
vals = np.array([[final_of(results[(pt, ts)]) for pt in PROFIT_GRID] for ts in TIME_STOP_GRID])
fig, ax = plt.subplots(figsize=(8, max(2.5, 1.2 * len(TIME_STOP_GRID))))
im = ax.imshow(np.log10(np.maximum(vals, 1)), cmap="viridis", aspect="auto")
ax.set_xticks(range(len(PROFIT_GRID)), [f"{int(p*100)}%" for p in PROFIT_GRID])
ax.set_yticks(range(len(TIME_STOP_GRID)), [f"{ts}min" for ts in TIME_STOP_GRID])
ax.set_xlabel("Profit target"); ax.set_ylabel("Time stop")
ax.set_title("Final balance (log10 color)  |  start $1,000")
for i in range(len(TIME_STOP_GRID)):
    for j in range(len(PROFIT_GRID)):
        ax.text(j, i, f"${vals[i, j]:,.0f}", ha="center", va="center",
                color="white", fontsize=8)
plt.colorbar(im, label="log10(final balance)")
plt.tight_layout(); plt.show()

## 6. Best combo — full breakdown + equity curve

In [ ]:
bt = results[best]
pt, tstop = best
if bt.empty:
    print("Best combo produced no trades.")
else:
    n      = len(bt)
    final  = bt["balance"].iloc[-1]
    wins   = bt[bt.pnl > 0]; losses = bt[bt.pnl <= 0]
    eq     = np.concatenate([[START_BALANCE], bt.balance.values])
    maxdd  = float(((eq - np.maximum.accumulate(eq)) / np.maximum.accumulate(eq)).min())
    bt2    = bt.copy(); bt2["ym"] = pd.to_datetime(bt2.date).dt.to_period("M")
    monthly = bt2.groupby("ym").pnl.sum()
    pf = wins.pnl.sum() / abs(losses.pnl.sum()) if losses.pnl.sum() != 0 else float("inf")

    print("=" * 54)
    print(f"  BEST: +{int(pt*100)}% profit / {tstop}min time stop")
    print("=" * 54)
    print(f"  Trades                 {n}")
    print(f"  Start / Final          ${START_BALANCE:,.0f}  ->  ${final:,.2f}")
    print(f"  Total return           {(final/START_BALANCE-1)*100:,.1f}%")
    print(f"  Win rate               {len(wins)/n*100:.1f}%  ({len(wins)} W / {len(losses)} L)")
    print(f"  Number of losers       {len(losses)}")
    print(f"  Avg trade PnL          ${bt.pnl.mean():,.2f}   ({bt.ret.mean()*100:+.2f}%)")
    print(f"  Avg winner / loser     ${(wins.pnl.mean() if len(wins) else 0):,.2f} / "
          f"${(losses.pnl.mean() if len(losses) else 0):,.2f}")
    print(f"  Profit factor          {pf:.2f}")
    print(f"  Max drawdown           {maxdd*100:.1f}%")
    print(f"  Positive months        {int((monthly>0).sum())} / {len(monthly)}")
    print(f"  Total commissions      ${(bt.contracts.sum()*COMMISSION*2):,.2f}")
    print(f"  Calls / Puts           {int((bt.cp=='C').sum())} / {int((bt.cp=='P').sum())}")
    print(f"  Exit breakdown         {bt.reason.value_counts().to_dict()}")
    print("=" * 54)

    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    ax[0].plot([bt.date.iloc[0]] + list(bt.date), list(eq))
    ax[0].set_yscale("log"); ax[0].set_title("Equity curve — best combo (log)")
    ax[0].set_ylabel("Balance ($)"); ax[0].grid(True, which="both", alpha=0.3)
    cols = ["#2ca02c" if v > 0 else "#d62728" for v in monthly.values]
    ax[1].bar([str(p) for p in monthly.index], monthly.values, color=cols)
    ax[1].set_title("Monthly PnL ($) — best combo"); ax[1].tick_params(axis="x", rotation=90)
    ax[1].grid(True, axis="y", alpha=0.3)
    plt.tight_layout(); plt.show()

    display(bt.assign(ret=(bt.ret*100).round(2)).round(2).tail(20))